# ATLOP Baseline 재현 (Revised 데이터, Colab A100)

> **최종 업데이트**: 2026-07-15 (**PUATLoss(na_weight=0.7)를 revised 데이터셋에 실험 적용 +
> early stopping(patience=5) + best-epoch 체크포인트 저장 로직 추가; 방금 완료된 순수
> baseline 실행을 revised 데이터셋의 baseline으로 확정**):
>
> **1) 방금 완료된 실행 결과 기록** (`run_name=atlop_baseline_revised`, PUATLoss/early
> stopping/best-checkpoint 전부 없는 순수 baseline, 셀 5의 실제 로그 기준): 마지막(20번째,
> epoch 19) dev F1 73.67/Ign F1 72.95, test F1 73.48/Ign F1 72.77 (P 87.54/R 63.31). **peak는
> epoch 14였음**(dev F1 73.97/Ign F1 73.14) — 당시엔 best-checkpoint 저장 로직이 없어 그
> 체크포인트 자체는 저장되지 못하고 마지막 epoch(19) 것만 남음, 최종 성적과 차이는
> 미미(−0.30/−0.19)하지만 정확히 이 문제(비단조 dev F1 때문에 최고점을 놓침)를 막기 위해
> 이번 업데이트로 best-checkpoint 저장을 추가함(아래 3번). 이 수치를 **revised 데이터셋의
> (plain ATLoss) baseline**으로 맨 아래 "결과 기록" 표에 기입함 — `dev_revised.json`/
> `test_revised.json`(각 500문서) 기준이라 원본 61.71/59.86과 직접 비교 불가하다는 건
> 기존과 동일.
>
> **2) PUATLoss(na_weight=0.7) 실험 적용 — ⚠️ 이 데이터셋에서 검증된 값은 아님**: 원본
> 데이터에서는 `ATLOP + PUATLoss(0.7)`이 검증됐지만(트랙3 `atlop_full_pu07`, dev F1
> 62.06/Ign F1 60.16 — plain ATLoss 대비 +0.35/+0.30, `results/comparison.md`), **revised
> 데이터셋에도 그대로 통할 거라는 근거는 없음**을 확인함 — `docred_data/data/
> relations_revised.csv`를 직접 까본 결과, `train_revised.json`의 relation label 중
> `evidence_source=annotated`(사람 검증)는 53.0%뿐이고 나머지는 `unresolved_multihop`
> (28.8%, 근거 문장 없이 자동 추론된 다중홉 합성) + `inferred_cooccurrence`(18.2%, 공출현
> 휴리스틱 추론)임(`confidence` 컬럼은 전부 1.0이라 신뢰도 구분에 못 씀). 즉 이 데이터셋의
> 잠재적 노이즈는 **positive label 쪽 과다추정**(false positive 위험)이지, na_weight=0.7이
> 원래 겨냥하는 **distant supervision의 Na(negative) 쪽 노이즈**(원본 train_distant에서
> 실측된 "dev 정답의 62.2%가 Na로 잘못 라벨링됨")와 메커니즘이 다름 — 그대로 가져다 쓴
> 0.7이 이 데이터셋에서도 최적이라고 볼 근거가 없으므로, 이번 실행은 **검증이 아니라
> plain 베이스라인(1번)과 비교하는 하나의 실험**으로 취급할 것(결과 표 참고, 개선이 확인돼야
> 새 기준으로 승격). `train_revised.json` 구성엔 distant 단계가 없어 stage 2 하나만
> 도는데, `--use_pu_loss`는 이 유일한 스테이지에 직접 PUATLoss를 적용함(distant 단계가
> 있는 `train_gat.py`의 기존 관례처럼 "distant 전용"이 아니라, 이번 데이터 구성에서 PU
> loss를 적용할 지점이 이 스테이지뿐이기 때문). 참고로 방금 완료된 plain 베이스라인의
> F1(73.67)이 원본 baseline(61.71)보다 훨씬 높게 나온 것도 이 자동 라벨링과 무관하지
> 않을 수 있음 — train/dev label이 같은 휴리스틱으로 만들어졌다면 모델이 "휴리스틱 재현"을
> 학습해 점수가 부풀려졌을 가능성도 있어, 절대 수치 해석에 참고할 것.
>
> **3) early stopping(patience=5) + best-epoch 체크포인트 저장**: `train_gat.py`에 이미 있던
> 것과 동일한 패턴(매 epoch dev F1 갱신 시 `{run_name}_best.pt`/
> `{run_name}_best_dev_predictions.json` 저장, `--early_stop_patience` 연속 미개선 시 조기
> 종료)을 `train_baseline.py`에 새로 포팅 — 위 1번에서 본 것처럼 "마지막 epoch이 최고가
> 아닐 수 있다"는 문제를 이제부터 방지함(이건 데이터셋과 무관하게 항상 유효). `--early_stop_patience 5`로 설정.
>
> 다음 Colab 실행(`run_name=atlop_pu07_revised`)부터 반영 — CPU smoke test(`--limit_docs 6
> --epochs 3 --use_pu_loss --early_stop_patience 2`)로 크래시 없이 PUATLoss/조기종료/
> best-checkpoint 저장 동작 확인 완료, 실측(GPU)은 아직.
>
> 이전 (**dk EGAT 실험 중단, ATLOP baseline 재현 파이프라인으로 원복**):
> `dk_gat` 트랙(그래프 증강)에서 revised 데이터로 재학습을 돌리다가, **원래 검증된 baseline
> 아키텍처**(BERT-base-cased + Sliding Window + Entity Marker/logsumexp Pooling + Localized
> Context Pooling + `[Entity;Context]→Linear→Tanh(1536→768)` + Grouped Bilinear + ATLoss +
> Adaptive Thresholding, 원본 데이터 기준 dev F1 61.71/Ign F1 59.86 검증됨)로 되돌리기로 함 —
> GAT/그래프 관련 구성 요소(Gated Fusion, Meta-path Attention, Entity-Pair Graph, Evidence
> Contrastive Loss 등)는 전부 제거. **데이터셋은 바로 이전과 동일하게 유지** —
> `train_revised.json`(distant 없음, 최대 20 epoch), `dev_revised.json`으로 매 epoch 검증,
> `test_revised.json`으로 최종 트리플 예측.
>
> 이 baseline은 `Scripts/atlop/re_model.py`/`preprocess.py`와 **정확히 동일한 수식**으로
> 재구현했지만(`Scripts/dk_gat/model_baseline.py`/`preprocess_baseline.py`), `Scripts/atlop`는
> 팀원 트랙이라 그 파일들을 직접 수정/import하지 않고 `Scripts/dk_gat/` 안에 완전히 자체
> 구현함 — 상세는 `Scripts/dk_gat/README.md`의 관할 구분 참고. 학습 스크립트도
> `Scripts.dk_gat.train_baseline`으로 교체.
>
> 참고로 바로 이전 GAT 실행(`dk_gat_revised`, 이 노트북에서 중단됨)의 로그는 epoch 3까지
> dev F1 61.50/Ign F1 60.71까지 순조롭게 올라가고 있었음 — encoder unfreeze(epoch 1) 이후
> 정상 궤도였던 것으로 보임. 이번 baseline 결과와 비교해볼 수 있도록 남겨둠.
>
> ⚠️ **주의**: `dev_revised.json`/`test_revised.json`은 각각 500문서로, 원본 baseline
> 61.71/59.86은 원본 `dev.json`(998문서) 기준이라 **직접 비교 불가** — 이번 실행 결과는 새
> 표로 따로 기록할 것 (맨 아래 "결과 기록" 셀 참고).

**실행 전**: 런타임 → 런타임 유형 변경 → **A100 GPU**.

**학습 구성**: `train_revised.json`(3,053개, distant 없음) × 최대 **20 epoch**,
**PUATLoss(na_weight=0.7, 이 데이터셋에서는 미검증 — 실험)** 학습, dev F1이 **5 epoch** 연속
개선 없으면 조기 종료(`--early_stop_patience 5`), `dev_revised.json`으로 매 epoch 검증 +
best-epoch 체크포인트 갱신, `test_revised.json`으로 (best-checkpoint 기준) 최종 트리플 예측 +
F1/Ign F1 산출. 모델은 BERT-base-cased + Entity Marker/logsumexp Pooling + Localized Context
Pooling + Grouped Bilinear Classifier + Adaptive Thresholding (그래프/GAT 없음).


In [7]:
# 0) GPU 확인 (A100이 보여야 함)
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-e517e286-953a-2010-04a8-00f7d5898935)


In [8]:
# 1) 코드 + 데이터 (Git LFS에서 필요한 json만 선별 pull)
#    Scripts/dk_gat는 아직 main이 아니라 dk 브랜치에만 있으므로 반드시 checkout 필요
#    distant supervision을 안 쓰므로 train_distant.json(417MB)은 pull하지 않음
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/multiful/Information_Extraction.git
%cd Information_Extraction
!git checkout dk
!git lfs pull --include="docred_data/data/train_revised.json,docred_data/data/dev_revised.json,docred_data/data/test_revised.json,docred_data/data/rel_info.json"
# json들이 KB~MB 단위로 보이면 성공 (133바이트면 LFS pull 실패)
!ls -lh docred_data/data/*revised* docred_data/data/rel_info.json
# Scripts/dk_gat가 실제로 있는지 확인 (여기 안 뜨면 아래 학습 셀에서 ModuleNotFoundError 남)
!ls Scripts/dk_gat/

Cloning into 'Information_Extraction'...
remote: Enumerating objects: 547, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 547 (delta 33), reused 31 (delta 28), pack-reused 498 (from 1)
Receiving objects: 100% (547/547), 42.60 MiB | 12.89 MiB/s, done.
Resolving deltas: 100% (307/307), done.
/content/Information_Extraction/Information_Extraction
Branch 'dk' set up to track remote branch 'dk' from 'origin'.
Switched to a new branch 'dk'
-rw-r--r-- 1 root root 3.1M Jul 15 02:36 docred_data/data/dev_revised.json
-rw-r--r-- 1 root root 2.4K Jul 15 02:36 docred_data/data/rel_info.json
-rw-r--r-- 1 root root 3.1M Jul 15 02:36 docred_data/data/test_revised.json
-rw-r--r-- 1 root root  18M Jul 15 02:36 docred_data/data/train_revised.json
colab_gat_a100.ipynb  model.py		      README.md
__init__.py	      preprocess_baseline.py  train_baseline.py
model_baseline.py     preprocess_gat.py       train_gat.py


In [3]:
# 2) 패키지 (모델은 허브가 아니라 로컬 파일에서 로드할 거라 aria2도 여기서 같이 설치)
!pip install -q transformers==4.57.6 accelerate
!apt-get -qq install -y aria2 > /dev/null

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
%%bash
# 3) bert-base-cased를 로컬로 직접 받기 (허브 다운로더 대신 aria2c 사용)
#    Colab <-> HF CDN 네트워크가 불안정해서 순정 다운로더는 느리게라도 버텼지만(최저 158kB/s),
#    hf_transfer는 세그먼트 하나가 끊기면 재시도 없이 그대로 멈춰버리는 것으로 확인됨
#    (65MB/s로 빠르게 가다 15%에서 하드행). aria2c는 -x/-s로 멀티커넥션, 끊기면 그 세그먼트만
#    재시도(--max-tries, --retry-wait)하므로 이 네트워크 환경에서 훨씬 안정적.
#    exit 22(HTTP response header bad)를 겪은 적 있어서 세 가지 보강:
#    (a) Colab은 세션마다 /content가 초기화되므로 --continue로 이어받다 손상된 부분파일과
#        충돌하는 걸 막기 위해 매번 디렉토리를 비우고 새로 받음.
#    (b) 커넥션 수를 16->8로 낮춰 HF CDN 레이트리밋/커넥션 리셋 가능성을 줄임.
#    (c) set -e만으로는 "일부 파일이 안 받아진 채로 학습 셀이 그대로 실행되는" 사고를
#        못 막으므로(루프 중간에 실패하면 그 시점까지 받은 파일만 남음), 루프 뒤에
#        파일별 최소 용량 검증을 추가해 하나라도 모자라면 이 셀 자체를 실패시킴.
set -e
rm -rf /content/bert-base-cased
mkdir -p /content/bert-base-cased
BASE_URL="https://huggingface.co/bert-base-cased/resolve/main"
for fname in model.safetensors config.json vocab.txt tokenizer_config.json tokenizer.json; do
  aria2c -x 8 -s 8 -k 1M --max-tries=20 --retry-wait=3 --continue=false \
    -d /content/bert-base-cased -o "$fname" \
    "$BASE_URL/$fname"
done

echo "--- 받은 파일 확인 ---"
ls -lh /content/bert-base-cased

python3 - <<'PY'
import os, sys
min_size = {
    "model.safetensors": 400_000_000,
    "config.json": 200,
    "vocab.txt": 100_000,
    "tokenizer_config.json": 10,
    "tokenizer.json": 300_000,
}
base = "/content/bert-base-cased"
bad = []
for fname, min_bytes in min_size.items():
    path = os.path.join(base, fname)
    size = os.path.getsize(path) if os.path.exists(path) else 0
    if size < min_bytes:
        bad.append(f"{fname}: {size}B (기대 최소 {min_bytes}B)")
if bad:
    print("다운로드 불완전 - 아래 파일이 부족합니다. 이 셀을 다시 실행하세요:", file=sys.stderr)
    for line in bad:
        print(" -", line, file=sys.stderr)
    sys.exit(1)
print("모든 파일 크기 정상 확인 완료.")
PY


07/15 01:41:32 [NOTICE] Downloading 1 item(s)

07/15 01:41:32 [NOTICE] CUID#7 - Redirecting to https://cas-bridge.xethub.hf.co/xet-bridge-us/621ffdc036468d709f174331/83c31be240458b001866527feebc3cece210a4aec957064b2f166d2dd6e8471f?Expires=1784083292&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzYyMWZmZGMwMzY0NjhkNzA5ZjE3NDMzMS84M2MzMWJlMjQwNDU4YjAwMTg2NjUyN2ZlZWJjM2NlY2UyMTBhNGFlYzk1NzA2NGIyZjE2NmQyZGQ2ZTg0NzFmKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDA4MzI5Mn19fV19&Signature=MEYCIQCXMKMC%7ExaJUZF3RabrgUZIsPH254JBplMF62Zld7rn9gIhAN2ygRm6Hn98g6imTQCcEht3Hk0ay5gCjk-AFGPf%7Eone&Key-Pair-Id=K3EPXBYC3CKDRZ&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&X-Xet-Cas-Uid=public&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=cas%2F20260715%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20260715T014132Z&X-Amz-Expires=3600&X-Amz-SignedHeade

In [ ]:
# 4) 학습: ATLOP baseline + PUATLoss(na_weight=0.7, 실험 -- 아래 참고) + early
#    stopping(patience=5) + best-epoch 체크포인트 저장, train_revised.json만 사용 (distant 없음)
#    --model_name_or_path/--train_split/--dev_split/--test_file/--distant_epochs 0은
#    이전 atlop_baseline_revised 실행과 동일한 의미 (load_docs()가 named split이 아니면 경로로 취급).
#    --use_pu_loss --na_weight 0.7: 트랙3 atlop_full_pu07(원본 데이터, dev F1 62.06/Ign F1 60.16 --
#    plain ATLoss 대비 +0.35/+0.30)에서 검증된 값을 그대로 가져온 것 -- 단, 이 값은 원본
#    train_distant의 Na(negative) 라벨 노이즈를 실측해서 나온 값이고, revised 데이터는
#    relations_revised.csv 기준 라벨의 47%가 자동 추론(unresolved_multihop/
#    inferred_cooccurrence, positive 쪽 과다추정 위험)이라 노이즈 성격 자체가 다름 --
#    0.7이 이 데이터셋에도 최적이라는 근거는 없어서 이번 실행은 "검증"이 아니라 plain
#    베이스라인(atlop_baseline_revised, dev F1 73.67/Ign F1 72.95)과 비교하는 실험으로 취급.
#    --early_stop_patience 5: dev F1이 5 epoch 연속 개선 없으면 조기 종료 + best-epoch 체크포인트
#    자동 저장({run_name}_best.pt) -- 직전 atlop_baseline_revised 실행에서 peak가 마지막 epoch이
#    아니라 epoch 14였던 것(dev F1 73.97 vs 최종 epoch 19의 73.67)을 놓치지 않기 위함(이건
#    데이터셋과 무관하게 항상 유효).
#    Scripts.dk_gat.train_baseline은 Scripts/atlop/re_model.py와 수식이 동일한
#    model_baseline.py(entity marker+logsumexp pooling, LCP, grouped bilinear, ATLoss/PUATLoss)를
#    쓰지만 atlop 파일을 직접 import하지 않고 dk_gat 안에 자체 구현한 것 -- README.md 참고.
!python -m Scripts.dk_gat.train_baseline \
  --model_name_or_path /content/bert-base-cased \
  --train_split docred_data/data/train_revised.json \
  --dev_split docred_data/data/dev_revised.json \
  --test_file docred_data/data/test_revised.json \
  --distant_epochs 0 --epochs 20 \
  --use_pu_loss --na_weight 0.7 --early_stop_patience 5 \
  --run_name atlop_pu07_revised --save_model --seed 66

In [ ]:
# 5) 결과 백업 (세션 끊기면 파일이 사라지므로 반드시 실행)
#    이번 실행부터 best-epoch 체크포인트/예측도 생성됨({run_name}_best.pt,
#    {run_name}_best_dev_predictions.json) -- 마지막 epoch 것과 함께 둘 다 백업.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"
!cp results/atlop_pu07_revised.pt \
   results/atlop_pu07_revised_best.pt \
   results/atlop_pu07_revised_dev_predictions.json \
   results/atlop_pu07_revised_best_dev_predictions.json \
   results/atlop_pu07_revised_test_predictions.json \
   "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"
!ls -lh "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/" | grep atlop_pu07_revised

## 결과 기록

**주의**: 이 실행들은 `dev_revised.json`/`test_revised.json`(각 500문서) 기준이라 원본 baseline
61.71/59.86(원본 `dev.json`, 998문서 기준)과 직접 비교할 수 없음 -- 아래처럼 새 표로 따로
관리함.

| 모델 | epochs | dev F1 | dev Ign F1 | test F1 | test Ign F1 |
|---|---|---|---|---|---|
| **ATLOP baseline 재현 (revised, `run_name=atlop_baseline_revised`)** — revised 데이터셋 baseline | 20 (early stop 없음, 마지막 epoch 기준) | 73.67 | 72.95 | 73.48 | 72.77 |
| ATLOP + PUATLoss(0.7, **이 데이터셋 미검증 — 실험**) + early stop(5) + best-epoch (revised, `run_name=atlop_pu07_revised`) | (실행 후 기입) | | | | |

- seed 66 단일 시드.
- baseline 행은 2026-07-15 Colab A100 실측(셀 5 로그 기준) — dev F1이 마지막 epoch(19)까지
  단조 증가하지 않았고 **실제 peak는 epoch 14(dev F1 73.97 / Ign F1 73.14)** 였음(최종 대비
  −0.30/−0.19, 미미하지만 이 문제를 막기 위해 이후 실행부터 best-epoch 체크포인트 저장을
  추가함 -- 맨 위 "최종 업데이트" 참고). 당시엔 저장 로직이 없어 그 체크포인트 자체는
  남지 않음.
- **PUATLoss 행의 na_weight=0.7은 원본 데이터(트랙3 `atlop_full_pu07`)에서 검증된 값을
  그대로 가져온 것으로, revised 데이터셋에서 최적이라는 근거는 없음** — `relations_revised.csv`
  확인 결과 label의 47%가 자동 추론(`unresolved_multihop`/`inferred_cooccurrence`, positive
  쪽 과다추정 위험)이라, na_weight=0.7이 원래 겨냥하는 distant supervision의 Na(negative)
  쪽 노이즈와 성격이 다름(맨 위 "최종 업데이트" 2번 참고). 이 행은 baseline 대비 **개선을
  확인해야 새 기준으로 승격**할 실험 결과로 취급할 것 — 개선 폭이 작거나 없으면 na_weight을
  스윕(예: 0.5/0.9)해보거나 이 데이터셋엔 PU loss 자체를 적용하지 않는 편이 나을 수 있음.
- 그래프/GAT 없이 순수 baseline만 재현한 결과이므로, 이후 dk_gat 트랙(GAT 증강)과 비교할 때
  baseline 행을 기준으로 삼을 것.
- 추가 실험(하이퍼파라미터 변경)은 이 노트북 셀 4의 커맨드 인자만 바꿔 반복하면 됨 -- 전체
  옵션은 `python -m Scripts.dk_gat.train_baseline --help` 참고.
